# Task Arithmetic — Treino Centralizado

Este notebook calibra a máscara (task arithmetic) e faz **treino centralizado** no CIFAR-100 usando `SparseSGDM` com a máscara.

Se você quiser comparar, rode também o notebook IID (FedAvg) e compare as curvas.

In [1]:
# --- 0) Runtime check ---
import os, sys
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

torch: 2.9.0+cu126
cuda available: True
gpu: Tesla T4


In [2]:
# --- 1) Clone repo (task_arithmetic branch) ---
!git clone -b task_arithmetic --single-branch https://github.com/caiopenayo/Federated-Learning-Under-the-Lens-of-Task-Arithmetic.git
%cd Federated-Learning-Under-the-Lens-of-Task-Arithmetic
!git pull origin task_arithmetic


# Make repo importable
repo_root = os.path.abspath(".")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

Cloning into 'Federated-Learning-Under-the-Lens-of-Task-Arithmetic'...
remote: Enumerating objects: 166, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 166 (delta 5), reused 5 (delta 5), pack-reused 154 (from 1)
Receiving objects: 100% (166/166), 241.01 KiB | 8.93 MiB/s, done.
Resolving deltas: 100% (57/57), done.
/content/Federated-Learning-Under-the-Lens-of-Task-Arithmetic
From https://github.com/caiopenayo/Federated-Learning-Under-the-Lens-of-Task-Arithmetic
 * branch            task_arithmetic -> FETCH_HEAD
Already up to date.


In [3]:
# --- 1) Make repo importable (local + Colab-friendly) ---
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print('repo_root:', repo_root)

USE_DRIVE = False
try:
    import google.colab  # type: ignore
    from google.colab import drive  # type: ignore
    USE_DRIVE = True
    drive.mount('/content/drive')
except Exception:
    USE_DRIVE = False

print('USE_DRIVE:', USE_DRIVE)

repo_root: /content
Mounted at /content/drive
USE_DRIVE: True


In [4]:
# --- 2) Imports ---
from optim.fisher import calibrate_gradient_mask_multi_round, MaskCalibrationConfig
from optim.sparse_sgdm import SparseSGDM
from models.vit_dino import build_dino_vit
from data.datasets import get_cifar100, get_cifar100_transforms
from data.partition import make_dataset_loaders

import torch.nn as nn
from torch.utils.data import DataLoader
import copy
import numpy as np

In [5]:
# --- 3) Model + data ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_dino_vit(num_classes=100, img_size=160).to(device)
criterion = nn.CrossEntropyLoss()
print('model ready on', device)

transform_train, transform_test = get_cifar100_transforms(img_size=160)
train, val, test = get_cifar100(
    train_transform=transform_train,
    test_transform=transform_test,
    val_ratio=0.1,
    root='./data',
    seed=42,
)
train_loader, val_loader, test_loader = make_dataset_loaders(train, val, test)
calib_loader = DataLoader(val, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)

print('train/val/test:', len(train), len(val), len(test))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/86.7M [00:00<?, ?B/s]

model ready on cuda


100%|██████████| 169M/169M [00:19<00:00, 8.64MB/s]


train/val/test: 45000 5000 10000


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


## Calibração da máscara (Task Arithmetic)

## Treino centralizado com `SparseSGDM` + máscara

A máscara aqui é aplicada pelo `SparseSGDM` usando `set_mask_from_named_params(model.named_parameters(), mask)` (máscara por **nome do parâmetro**).

In [ ]:
# --- 5) Central training loop (checkpoint + resume) ---
import os
import copy
import torch

def evaluate_acc(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            logits = model(x)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.numel()
    return 100.0 * correct / max(1, total)


def save_ckpt(path, epoch, model, optimizer, history):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save(
        {
            "epoch": epoch,
            "model_state": model.state_dict(),
            "opt_state": optimizer.state_dict(),
            "history": history,
        },
        path,
    )


def load_ckpt(path, model, optimizer, map_location):
    ckpt = torch.load(path, map_location=map_location)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["opt_state"])
    history = ckpt.get("history", {"epoch": [], "test_acc": [], "train_loss": []})
    start_epoch = int(ckpt.get("epoch", 0)) + 1
    return start_epoch, history


def compute_mask_stats(mask_dict):
    total, trainable = 0, 0
    for m in mask_dict.values():
        total += m.numel()
        trainable += int((m != 0).sum())
    return total, trainable, trainable / max(1, total)


def train_centralized(
    base_model,
    train_loader,
    test_loader,
    *,
    device,
    epochs=50,
    lr=0.01,
    log_every=1,
    mask=None,
    resume=False,
    ckpt_path=None,
    ckpt_every=5,
    weight_decay=0.0,
    momentum=0.0,
    nesterov=False,
    grad_clip_norm=None,
    ):
    assert ckpt_path is not None, "Provide ckpt_path for resume safety."
    model = copy.deepcopy(base_model).to(device)
    criterion = nn.CrossEntropyLoss()

    if mask is None:
        optimizer = torch.optim.SGD(
            model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay, nesterov=nesterov
        )
    else:
        optimizer = SparseSGDM(
            model.parameters(),
            lr=lr,
            momentum=momentum,
            weight_decay=weight_decay,
            nesterov=nesterov,
        )
        optimizer.set_mask_from_named_params(model.named_parameters(), mask)

    history = {"epoch": [], "test_acc": [], "train_loss": []}
    start_epoch = 1
    if resume and os.path.exists(ckpt_path):
        start_epoch, history = load_ckpt(ckpt_path, model, optimizer, map_location=device)
        print(f"[Resume] Loaded {ckpt_path} -> start_epoch={start_epoch}")

    for ep in range(start_epoch, epochs + 1):
        model.train()
        loss_sum = 0.0
        total = 0

        for x, y in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            if grad_clip_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
            optimizer.step()

            bs = x.size(0)
            loss_sum += loss.item() * bs
            total += bs

        train_loss = loss_sum / max(1, total)
        test_acc = evaluate_acc(model, test_loader, device)

        history["epoch"].append(ep)
        history["train_loss"].append(train_loss)
        history["test_acc"].append(test_acc)

        if ep == 1 or (log_every is not None and log_every > 0 and ep % log_every == 0):
            print(f"Epoch {ep:03d}/{epochs} | train_loss {train_loss:.4f} | test_acc {test_acc:.2f}%")

        if ckpt_every is not None and ckpt_every > 0 and ep % ckpt_every == 0:
            save_ckpt(ckpt_path, ep, model, optimizer, history)

    save_ckpt(ckpt_path, epochs, model, optimizer, history)
    return history


if USE_DRIVE:
    ckpt_dir = "/content/drive/MyDrive/central_ckpts"
else:
    ckpt_dir = os.path.abspath(os.path.join(repo_root, "ckpts_central"))
os.makedirs(ckpt_dir, exist_ok=True)

EPOCHS = 25
LR = 0.01
trainable_fraction_list = [0.1, 0.20]
calib_rounds_list = [1, 3, 5]
fisher_batches_per_round = 5
rule = 'least_sensitive'

for trainable_fraction in trainable_fraction_list:
  for calib_rounds in calib_rounds_list:
    exp_name = f"ta_central_tf{trainable_fraction}_cr{calib_rounds}_E{EPOCHS}_LR{LR}"
    ckpt_path = os.path.join(ckpt_dir, exp_name + ".pth")
    print("ckpt_path:", ckpt_path)

    # --- 4) Mask calibration ---

    cfg = MaskCalibrationConfig(
        trainable_fraction=trainable_fraction,
        rounds=calib_rounds,
        fisher_batches_per_round=fisher_batches_per_round,
        rule=rule,
    )

    mask = calibrate_gradient_mask_multi_round(
        model=copy.deepcopy(model),
        dataloader=calib_loader,
        criterion=criterion,
        device=device,
        cfg=cfg,
    )

    total, trainable, ratio = compute_mask_stats(mask)
    print(f'Mask stats: trainable params {trainable:,}/{total:,} ({ratio:.2%})')

    hist = train_centralized(
        base_model=model,
        train_loader=train_loader,
        test_loader=test_loader,
        device=device,
        epochs=EPOCHS,
        lr=LR,
        log_every=1,
        mask=mask,
        resume=True,
        ckpt_path=ckpt_path,
        ckpt_every=5,
        weight_decay=0.0,
        momentum=0.0,
        nesterov=False,
        grad_clip_norm=1.0,
    )

    hist.keys(), {k: len(v) for k, v in hist.items()}

ckpt_path: /content/drive/MyDrive/central_ckpts/ta_central_tf0.1_cr1_E25_LR0.01.pth
Mask stats: trainable params 2,166,730/21,667,300 (10.00%)


In [ ]:
# --- 6) Plot ---
import matplotlib.pyplot as plt

plt.figure()
plt.plot(hist['epoch'], hist['test_acc'])
plt.title('Centralized — Test Acc vs Epoch')
plt.xlabel('Epoch')
plt.ylabel('Test Accuracy (%)')
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure()
plt.plot(hist['epoch'], hist['train_loss'])
plt.title('Centralized — Train Loss vs Epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.tight_layout()
plt.show()